# 3.5 Lab: Why Standard Attention is Memory-Bound

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.5_flash_attention_problem/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.5_flash_attention_problem/lab.ipynb)

This lab demonstrates two core observations:
1. The attention matrix grows quadratically with sequence length
2. Standard attention is memory-bound on modern GPUs (arithmetic intensity below ridge point)


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# GPU specs for A100 80GB SXM
A100_HBM_GB = 80           # HBM capacity in GB
A100_BW_TBs = 2.0          # HBM bandwidth in TB/s
A100_TFLOPS = 312          # Peak FP16 TFLOPS
A100_SRAM_MB = 20          # Total on-chip SRAM in MB
RIDGE_POINT = A100_TFLOPS * 1e3 / (A100_BW_TBs * 1e3)  # FLOPs/byte


## Experiment 1: Quadratic Memory Growth

The attention matrix has shape [N, N] per head. We compute its size across sequence lengths and compare to GPU memory capacity.


In [ ]:
# --- Parameters (change these and re-run) ---
SEQ_LENGTHS = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
NUM_HEADS = 32        # Typical for 7B models
HEAD_DIM = 128        # Typical for modern architectures
DTYPE_BYTES = 2       # FP16

# --- Compute attention matrix sizes ---
# Each head materializes one [N, N] matrix for S and one for P
attn_matrix_bytes = [n * n * DTYPE_BYTES for n in SEQ_LENGTHS]  # per head
total_bytes = [b * NUM_HEADS * 2 for b in attn_matrix_bytes]    # S + P, all heads
total_gb = [b / 1e9 for b in total_bytes]

# --- Plot: Attention matrix size vs sequence length ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.semilogy(SEQ_LENGTHS, total_gb, 'o-', color='#2563eb', linewidth=2, markersize=8)
ax.axhline(y=A100_HBM_GB, color='#991b1b', linestyle='--', linewidth=2, label=f'A100 HBM capacity ({A100_HBM_GB} GB)')
ax.axhline(y=A100_SRAM_MB / 1000, color='#166534', linestyle=':', linewidth=2, label=f'A100 SRAM total ({A100_SRAM_MB} MB)')
ax.set_xlabel('Sequence Length (N)', fontsize=12)
ax.set_ylabel('Intermediate Memory (GB, log scale)', fontsize=12)
ax.set_title('Attention Intermediates: Quadratic Memory Growth\n(32 heads, FP16, S + P matrices)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(SEQ_LENGTHS)
ax.set_xticklabels([str(n) for n in SEQ_LENGTHS], rotation=45)
plt.tight_layout()
plt.show()

# Print the crossover point
for n, gb in zip(SEQ_LENGTHS, total_gb):
    marker = " <<< EXCEEDS HBM" if gb > A100_HBM_GB else ""
    print(f"  N={n:>6}: {gb:>8.3f} GB{marker}")


## Experiment 2: HBM Traffic Breakdown

We compute the exact byte count for reads and writes, separating linear (unavoidable) terms from quadratic (wasteful) terms.


In [ ]:
# --- HBM traffic calculation ---
def compute_hbm_traffic(n, d, num_heads, dtype_bytes=2):
    """Compute total HBM bytes for standard attention (all heads)."""
    # Linear terms: read Q,K,V + read V again + write O = 5*N*d per head
    linear_elements = 5 * n * d
    # Quadratic terms: write S + read S + write P + read P = 4*N^2 per head
    quadratic_elements = 4 * n * n
    # Total across all heads, in bytes
    total_elements = (linear_elements + quadratic_elements) * num_heads
    return {
        'linear_bytes': linear_elements * num_heads * dtype_bytes,
        'quadratic_bytes': quadratic_elements * num_heads * dtype_bytes,
        'total_bytes': total_elements * dtype_bytes,
        'quadratic_ratio': quadratic_elements / (linear_elements + quadratic_elements)
    }

# --- Compute for range of sequence lengths ---
results = [compute_hbm_traffic(n, HEAD_DIM, NUM_HEADS) for n in SEQ_LENGTHS]
linear_gb = [r['linear_bytes'] / 1e9 for r in results]
quadratic_gb = [r['quadratic_bytes'] / 1e9 for r in results]
ratios = [r['quadratic_ratio'] * 100 for r in results]

# --- Stacked bar chart: linear vs quadratic HBM traffic ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(SEQ_LENGTHS))
ax1.bar(x, linear_gb, color='#dcfce7', edgecolor='#000', label='Linear (unavoidable)')
ax1.bar(x, quadratic_gb, bottom=linear_gb, color='#ffe4e6', edgecolor='#000', label='Quadratic (wasteful)')
ax1.set_xticks(x)
ax1.set_xticklabels([str(n) for n in SEQ_LENGTHS], rotation=45)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('HBM Traffic (GB)')
ax1.set_title('HBM Traffic: Linear vs Quadratic')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# --- Pie chart for N=4096 ---
r = compute_hbm_traffic(4096, HEAD_DIM, NUM_HEADS)
ax2.pie([r['linear_bytes'], r['quadratic_bytes']],
        labels=['Linear (Q,K,V,O)', 'Quadratic (S,P)'],
        colors=['#dcfce7', '#ffe4e6'], autopct='%1.1f%%',
        textprops={'fontsize': 12}, startangle=90,
        wedgeprops={'edgecolor': '#000'})
ax2.set_title(f'Traffic Split at N=4096\n({r["quadratic_ratio"]*100:.1f}% is wasteful)')

plt.tight_layout()
plt.show()


## Experiment 3: Arithmetic Intensity and the Roofline

We compute the arithmetic intensity of standard attention and show it falls below the A100 ridge point, proving memory-boundedness.


In [ ]:
# --- Arithmetic intensity calculation ---
def compute_arithmetic_intensity(n, d, num_heads, dtype_bytes=2):
    """Compute FLOPs/byte for standard attention."""
    # FLOPs: Q@K^T and P@V are each 2*N*N*d per head
    flops_per_head = 2 * 2 * n * n * d  # two matmuls
    total_flops = flops_per_head * num_heads
    # Bytes: from traffic calculation
    traffic = compute_hbm_traffic(n, d, num_heads, dtype_bytes)
    intensity = total_flops / traffic['total_bytes']
    return intensity, total_flops, traffic['total_bytes']

# --- Compute intensity across sequence lengths ---
intensities = []
for n in SEQ_LENGTHS:
    ai, flops, bts = compute_arithmetic_intensity(n, HEAD_DIM, NUM_HEADS)
    intensities.append(ai)

# --- Roofline-style plot ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(SEQ_LENGTHS, intensities, 'o-', color='#2563eb', linewidth=2, markersize=8,
        label='Standard Attention')
ax.axhline(y=RIDGE_POINT, color='#991b1b', linestyle='--', linewidth=2,
           label=f'A100 Ridge Point ({RIDGE_POINT:.0f} FLOPs/byte)')
ax.fill_between(SEQ_LENGTHS, 0, RIDGE_POINT, alpha=0.08, color='#991b1b')
ax.text(SEQ_LENGTHS[-1], RIDGE_POINT * 0.7, 'MEMORY-BOUND REGION',
        ha='right', fontsize=11, color='#991b1b', fontstyle='italic')
ax.text(SEQ_LENGTHS[-1], RIDGE_POINT * 1.15, 'COMPUTE-BOUND REGION',
        ha='right', fontsize=11, color='#166534', fontstyle='italic')
ax.set_xlabel('Sequence Length (N)', fontsize=12)
ax.set_ylabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_title('Standard Attention: Always Memory-Bound on A100', fontsize=13)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
ax.set_xticks(SEQ_LENGTHS)
ax.set_xticklabels([str(n) for n in SEQ_LENGTHS], rotation=45)
plt.tight_layout()
plt.show()

# Print values
print(f"{'N':>8} | {'Intensity':>12} | {'Status'}")
print("-" * 40)
for n, ai in zip(SEQ_LENGTHS, intensities):
    status = "MEMORY-BOUND" if ai < RIDGE_POINT else "COMPUTE-BOUND"
    print(f"{n:>8} | {ai:>10.1f} | {status}")


## Key Takeaways

1. **Quadratic scaling**: Attention matrix memory grows as O(N^2), exceeding A100 HBM at ~32K tokens
2. **Wasteful traffic**: Over 96% of HBM traffic at N=4096 is from intermediate matrices that are used once and discarded
3. **Memory-bound**: Despite being pure matrix multiplication, standard attention never reaches the A100 ridge point because HBM traffic grows faster than useful FLOPs
4. **The opportunity**: Eliminating the N^2 intermediates (keeping them in SRAM) would make attention compute-bound, unlocking the GPU's full potential
